In [1]:

!gcc --version
!nproc
!ls -lah /kaggle/input/


gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

4
total 12K
drwxr-xr-x 3 root root 4.0K Oct  9 13:08 .
drwxr-xr-x 5 root root 4.0K Oct  9 13:08 ..
drwxr-xr-x 3 root root 4.0K Oct  9 13:08 datasets


In [2]:

from pathlib import Path

roots = list(Path("/kaggle/input").rglob("*"))
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

images = sorted(
    p for p in roots
    if p.is_file() and p.suffix.lower() in image_extensions
)

print("Images found:", len(images))
print("Sample image paths:")
for p in images[:5]:
    print(p)

assert images, "No images found. Check your uploaded Kaggle dataset."


Images found: 5000
Sample image paths:
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000139.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000285.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000632.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000724.jpg
/kaggle/input/datasets/xthink/coco-2017-val-images/val2017/000000000776.jpg


In [4]:

from PIL import Image
from pathlib import Path

WORK = Path("/kaggle/working/ssd_project")
PROCESSED = WORK / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

LIMIT = min(5000, len(images))
count = 0

for i, path in enumerate(images[:LIMIT]):
    try:
        with Image.open(path) as im:
            im.convert("L").resize((256, 256)).save(
                PROCESSED / f"{i:05d}.png"
            )
        count += 1
    except Exception as e:
        print("Skipped:", path.name, str(e)[:100])

print("Processed images:", count)
print("Output folder:", PROCESSED)

assert count >= 100, "At least 100 readable images are needed."


Processed images: 5000
Output folder: /kaggle/working/ssd_project/processed


In [6]:

import numpy as np
import pandas as pd
from PIL import Image

DATA = WORK / "data"
IMG_DIR = DATA / "set_100"
DATA.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

# Create a reproducible template
template = rng.integers(0, 256, (32, 32), dtype=np.uint8)
template.tofile(DATA / "template.bin")

processed = sorted(PROCESSED.glob("*.png"))
rng = np.random.default_rng(123)
ground_truth = []

for i, path in enumerate(processed[:100]):
    with Image.open(path) as im:
        img = np.array(im.convert("L"), dtype=np.uint8)

    planted = int(rng.random() < 0.30)
    x = int(rng.integers(0, 225)) if planted else -1
    y = int(rng.integers(0, 225)) if planted else -1

    if planted:
        img[y:y+32, x:x+32] = template

    img.tofile(IMG_DIR / f"{i:06d}.bin")
    ground_truth.append({
        "image_id": i, "planted": planted, "x": x, "y": y
    })

pd.DataFrame(ground_truth).to_csv(
    IMG_DIR / "ground_truth.csv", index=False
)

print("Template created:", (DATA / "template.bin").exists())
print("Binary test images:", len(list(IMG_DIR.glob("*.bin"))))
print("Ground truth saved.")


Template created: True
Binary test images: 100
Ground truth saved.


In [8]:

seq_code = r"""
#include <stdio.h>
#include <stdint.h>
#include <stdlib.h>
#include <inttypes.h>
#include <limits.h>

#define W 256
#define H 256
#define T 32

int main(int argc, char **argv) {
    if (argc != 4) {
        printf("Usage: %s DATA_DIR N OUTPUT.csv\n", argv[0]);
        return 1;
    }

    char path[1024];
    int n = atoi(argv[2]);
    uint8_t templ[T*T], img[W*H];

    snprintf(path, sizeof(path), "%s/../template.bin", argv[1]);
    FILE *f = fopen(path, "rb");
    if (!f || fread(templ, 1, T*T, f) != T*T) {
        fprintf(stderr, "Cannot read template.bin\n");
        if (f) fclose(f);
        return 1;
    }
    fclose(f);

    int *xs = calloc(n, sizeof(int));
    int *ys = calloc(n, sizeof(int));
    uint64_t *scores = calloc(n, sizeof(uint64_t));
    if (!xs || !ys || !scores) return 1;

    for (int id = 0; id < n; id++) {
        snprintf(path, sizeof(path), "%s/%06d.bin", argv[1], id);
        f = fopen(path, "rb");
        if (!f || fread(img, 1, W*H, f) != W*H) {
            fprintf(stderr, "Cannot read image: %s\n", path);
            if (f) fclose(f);
            return 1;
        }
        fclose(f);

        uint64_t best = UINT64_MAX;
        int bx = 0, by = 0;

        for (int y = 0; y <= H-T; y++) {
            for (int x = 0; x <= W-T; x++) {
                uint64_t ssd = 0;
                for (int ty = 0; ty < T; ty++) {
                    for (int tx = 0; tx < T; tx++) {
                        int d = (int)img[(y+ty)*W+x+tx]
                              - (int)templ[ty*T+tx];
                        ssd += (uint64_t)(d*d);
                    }
                }
                if (ssd < best) {
                    best = ssd;
                    bx = x; by = y;
                }
            }
        }
        xs[id] = bx; ys[id] = by; scores[id] = best;
    }

    f = fopen(argv[3], "w");
    if (!f) return 1;
    fprintf(f, "image_id,x,y,ssd\n");
    for (int i = 0; i < n; i++)
        fprintf(f, "%d,%d,%d,%" PRIu64 "\n",
                i, xs[i], ys[i], scores[i]);
    fclose(f);
    free(xs); free(ys); free(scores);
    return 0;
}
"""

(WORK / "ssd_seq.c").write_text(seq_code)
print("Sequential C source created.")


Sequential C source created.


In [11]:

import subprocess

def run_checked(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError("Command failed.")
    return r

run_checked([
    "gcc", "-O3", "-std=c11",
    str(WORK / "ssd_seq.c"),
    "-o", str(WORK / "ssd_seq")
])

run_checked([
    str(WORK / "ssd_seq"),
    str(IMG_DIR), "100",
    str(WORK / "sequential_100.csv")
])

print("Sequential SSD completed.")
print(pd.read_csv(WORK / "sequential_100.csv").head())


Sequential SSD completed.
   image_id    x    y      ssd
0         0  117  223  5441951
1         1  204   49        0
2         2   75   39        0
3         3  203  168  6085806
4         4   16   67  5518843


In [12]:

omp_code = r"""
#include <stdio.h>
#include <stdint.h>
#include <stdlib.h>
#include <inttypes.h>
#include <limits.h>
#include <omp.h>

#define W 256
#define H 256
#define T 32

int main(int argc, char **argv) {
    if (argc != 4) {
        printf("Usage: %s DATA_DIR N OUTPUT.csv\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[2]);
    char path[1024];
    uint8_t templ[T*T];

    snprintf(path, sizeof(path), "%s/../template.bin", argv[1]);
    FILE *f = fopen(path, "rb");
    if (!f || fread(templ, 1, T*T, f) != T*T) {
        fprintf(stderr, "Cannot read template.bin\n");
        if (f) fclose(f);
        return 1;
    }
    fclose(f);

    int *xs = calloc(n, sizeof(int));
    int *ys = calloc(n, sizeof(int));
    uint64_t *scores = calloc(n, sizeof(uint64_t));
    if (!xs || !ys || !scores) return 1;

    int failed = 0;
    #pragma omp parallel for schedule(static) reduction(|:failed) num_threads(4)
    for (int id = 0; id < n; id++) {
        uint8_t img[W*H];
        char filename[1024];
        snprintf(filename, sizeof(filename), "%s/%06d.bin", argv[1], id);

        FILE *in = fopen(filename, "rb");
        if (!in || fread(img, 1, W*H, in) != W*H) {
            if (in) fclose(in);
            failed = 1;
            continue;
        }
        fclose(in);

        uint64_t best = UINT64_MAX;
        int bx = 0, by = 0;

        for (int y = 0; y <= H-T; y++) {
            for (int x = 0; x <= W-T; x++) {
                uint64_t ssd = 0;
                for (int ty = 0; ty < T; ty++) {
                    for (int tx = 0; tx < T; tx++) {
                        int d = (int)img[(y+ty)*W+x+tx]
                              - (int)templ[ty*T+tx];
                        ssd += (uint64_t)(d*d);
                    }
                }
                if (ssd < best) {
                    best = ssd;
                    bx = x; by = y;
                }
            }
        }
        xs[id] = bx; ys[id] = by; scores[id] = best;
    }

    if (failed) {
        fprintf(stderr, "One or more images could not be read.\n");
        return 1;
    }

    f = fopen(argv[3], "w");
    if (!f) return 1;
    fprintf(f, "image_id,x,y,ssd\n");
    for (int i = 0; i < n; i++)
        fprintf(f, "%d,%d,%d,%" PRIu64 "\n",
                i, xs[i], ys[i], scores[i]);
    fclose(f);
    free(xs); free(ys); free(scores);
    return 0;
}
"""

(WORK / "ssd_omp.c").write_text(omp_code)

run_checked([
    "gcc", "-O3", "-std=c11", "-fopenmp",
    str(WORK / "ssd_omp.c"),
    "-o", str(WORK / "ssd_omp")
])

env = __import__("os").environ.copy()
env["OMP_NUM_THREADS"] = "4"

r = subprocess.run([
    str(WORK / "ssd_omp"), str(IMG_DIR), "100",
    str(WORK / "openmp_100.csv")
], capture_output=True, text=True, env=env)

if r.returncode:
    raise RuntimeError(r.stderr or r.stdout)

print("OpenMP SSD completed using 4 threads.")
print(pd.read_csv(WORK / "openmp_100.csv").head())


OpenMP SSD completed using 4 threads.
   image_id    x    y      ssd
0         0  117  223  5441951
1         1  204   49        0
2         2   75   39        0
3         3  203  168  6085806
4         4   16   67  5518843


In [15]:

import time
import os
from statistics import median

seq_exe = WORK / "ssd_seq"
omp_exe = WORK / "ssd_omp"
seq_csv = WORK / "sequential_100.csv"
omp_csv = WORK / "openmp_100.csv"

env = os.environ.copy()
env["OMP_NUM_THREADS"] = "4"

def timed_run(exe, output, parallel=False):
    start = time.perf_counter()
    r = subprocess.run(
        [str(exe), str(IMG_DIR), "100", str(output)],
        capture_output=True, text=True,
        env=env if parallel else None
    )
    elapsed = time.perf_counter() - start
    if r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return elapsed

# Warm-up
timed_run(seq_exe, seq_csv)
timed_run(omp_exe, omp_csv, True)

seq_times = [timed_run(seq_exe, seq_csv) for _ in range(3)]
omp_times = [timed_run(omp_exe, omp_csv, True) for _ in range(3)]

seq_time = median(seq_times)
omp_time = median(omp_times)

print("===== PERFORMANCE RESULTS =====")
print(f"Sequential time: {seq_time:.4f} seconds")
print(f"OpenMP time:     {omp_time:.4f} seconds")
print(f"Speedup:           {seq_time / omp_time:.2f}x")
print(f"Time saved:        {(seq_time-omp_time)/seq_time*100:.2f}%")


===== PERFORMANCE RESULTS =====
Sequential time: 2.2964 seconds
OpenMP time:     1.0739 seconds
Speedup:           2.14x
Time saved:        53.24%


In [16]:

seq = pd.read_csv(WORK / "sequential_100.csv")
omp = pd.read_csv(WORK / "openmp_100.csv")

cols = ["image_id", "x", "y", "ssd"]
same = seq[cols].equals(omp[cols])

print("Sequential and OpenMP outputs identical:", same)
assert same, "The two implementations produced different results."

gt = pd.read_csv(IMG_DIR / "ground_truth.csv")
check = gt.merge(seq, on="image_id", suffixes=("_true", "_found"))
planted = check[check["planted"] == 1]

if len(planted):
    correct = (
        (planted["x_true"] == planted["x_found"]) &
        (planted["y_true"] == planted["y_found"]) &
        (planted["ssd"] == 0)
    )
    print("Images with planted template:", len(planted))
    print("Correct detections:", int(correct.sum()))
    print(f"Detection accuracy: {correct.mean()*100:.2f}%")
else:
    print("No planted images to evaluate.")


Sequential and OpenMP outputs identical: True
Images with planted template: 34
Correct detections: 34
Detection accuracy: 100.00%


In [17]:

from sklearn.model_selection import train_test_split

processed_images = sorted(PROCESSED.glob("*.png"))

train, test = train_test_split(
    [str(p) for p in processed_images],
    test_size=0.20,
    random_state=42
)

pd.DataFrame({"image_path": train}).to_csv(
    WORK / "train_split.csv", index=False
)
pd.DataFrame({"image_path": test}).to_csv(
    WORK / "test_split.csv", index=False
)

print("Training images:", len(train))
print("Testing images:", len(test))
print("Saved train_split.csv and test_split.csv")


Training images: 4000
Testing images: 1000
Saved train_split.csv and test_split.csv
